In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/s5e10-lgbm-origcol-20seeds/oof_20seedslgb_plus_origcol.csv
/kaggle/input/s5e10-lgbm-origcol-20seeds/test_20seedslgb_plus_origcol.csv
/kaggle/input/s5e10-lgbm-origcol-20seeds/__results__.html
/kaggle/input/s5e10-lgbm-origcol-20seeds/__notebook__.ipynb
/kaggle/input/s5e10-lgbm-origcol-20seeds/__output__.json
/kaggle/input/s5e10-lgbm-origcol-20seeds/custom.css
/kaggle/input/s5e10-lgbm-origcol-20seeds/__results___files/__results___11_0.png
/kaggle/input/s5e10-xgb-origcol-20seeds/__results__.html
/kaggle/input/s5e10-xgb-origcol-20seeds/oof_xgb_plus_origcol.csv
/kaggle/input/s5e10-xgb-origcol-20seeds/__notebook__.ipynb
/kaggle/input/s5e10-xgb-origcol-20seeds/__output__.json
/kaggle/input/s5e10-xgb-origcol-20seeds/test_xgb_plus_origcol.csv
/kaggle/input/s5e10-xgb-origcol-20seeds/custom.css
/kaggle/input/s5e10-xgb-origcol-20seeds/__results___files/__results___11_0.png
/kaggle/input/pss5e10-main/lgb_20seed_oof_residuals.csv
/kaggle/input/pss5e10-main/cat_20seed_oof_residuals.csv
/

In [2]:
!pip install autogluon.tabular[0]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.1/225.1 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.2/64.2 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 66.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 487.3/487.3 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.0/278.0 kB 15.0 MB/s eta 0:00:00
  Attempting uninstall: psutil
    Found existing installation: psutil 7.1.0
    Uninstalling psutil-7.1.0:
      Successfully uninstalled psutil-7.1.0
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.2.2
    Uninstalling scikit-learn-1.2.2:
      Successfully uninstalled scikit-learn-1.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
category-encoders 2.7.

In [3]:
# train = pd.read_csv('/kaggle/input/pss5e10-main/final_train.csv')
# # train = train.fillna(0)
# test = pd.read_csv('/kaggle/input/pss5e10-main/test.csv')
# test = test.drop(columns='accident_risk')
# train['residual_risk'] = train['accident_risk'] - train['y']
# train.drop(columns='accident_risk', inplace=True)

In [4]:
oofs_df_baseline = pd.read_csv('/kaggle/input/pss5e10-oofs/oofs_autogluon_baseline.csv')
test_df_baseline = pd.read_csv('/kaggle/input/pss5e10-oofs/test_preds_autogluon_baseline.csv')
oofs_df_residuals = pd.read_csv('/kaggle/input/pss5e10-oofs/oofs_autogluon_residuals.csv')
test_df_residuals = pd.read_csv('/kaggle/input/pss5e10-oofs/test_preds_autogluon_residuals.csv')

oofs_df_residuals.columns = [col + '_res' for col in oofs_df_residuals.columns]
test_df_residuals.columns = [col + '_res' for col in test_df_residuals.columns]

oofs_df_tabm = pd.read_csv('/kaggle/input/s5e10-single-tabm-tuned/oof_tabm_plus_origcol_tuned.csv')
test_df_tabm = pd.read_csv('/kaggle/input/s5e10-single-tabm-tuned/test_tabm_plus_origcol_tuned.csv')

oofs_df_lgbm = pd.read_csv('/kaggle/input/s5e10-lgbm-origcol-20seeds/oof_20seedslgb_plus_origcol.csv')
test_df_lgbm = pd.read_csv('/kaggle/input/s5e10-lgbm-origcol-20seeds/test_20seedslgb_plus_origcol.csv')

oofs_df_xgb = pd.read_csv('/kaggle/input/s5e10-xgb-origcol-20seeds/oof_xgb_plus_origcol.csv')
test_df_xgb = pd.read_csv('/kaggle/input/s5e10-xgb-origcol-20seeds/test_xgb_plus_origcol.csv')

oofs_df_mlp = pd.read_csv('/kaggle/input/s5e10-realmlp-tuned/oof_realmlp_plus_origcol.csv')
test_df_mlp = pd.read_csv('/kaggle/input/s5e10-realmlp-tuned/test_realmlp_plus_origcol.csv')

oofs_df_tabm_res = pd.read_csv('/kaggle/input/s5e10-tabm-over-residuals/oof_tabm_overresid.csv')
test_df_tabm_res = pd.read_csv('/kaggle/input/s5e10-tabm-over-residuals/test_tabm_overresid.csv')

oofs_df_xgb2 = pd.read_csv('/kaggle/input/pss5e10-main/xgb_20seed_oof_residuals.csv')
test_df_xgb2 = pd.read_csv('/kaggle/input/pss5e10-main/xgb_20seed_test_residuals.csv')

oofs_df_cat2 = pd.read_csv('/kaggle/input/pss5e10-main/cat_20seed_oof_residuals.csv')
test_df_cat2 = pd.read_csv('/kaggle/input/pss5e10-main/cat_20seed_test_residuals.csv')

oofs_df_lgb2 = pd.read_csv('/kaggle/input/pss5e10-main/lgb_20seed_oof_residuals.csv')
test_df_lgb2 = pd.read_csv('/kaggle/input/pss5e10-main/lgb_20seed_test_residuals.csv')

oofs_df = pd.concat([
    oofs_df_tabm.drop(columns='id').add_prefix('tabm_'),
    oofs_df_lgbm.drop(columns='id').add_prefix('lgbm_'),
    oofs_df_xgb.drop(columns='id').add_prefix('xgb_'),
    oofs_df_mlp.drop(columns='id').add_prefix('mlp_'),
    oofs_df_tabm_res.drop(columns='id').add_prefix('tabm_res_'),
    # oofs_df_xgb2.drop(columns=['id', 'Unnamed: 0']).add_prefix('xgb2_'),
    oofs_df_cat2.drop(columns=['id', 'Unnamed: 0']).add_prefix('cat2_'),
    oofs_df_lgb2.drop(columns='id').add_prefix('lgb2_'),
    oofs_df_residuals['WeightedEnsemble_L5_res'],
], axis=1)

test_df = pd.concat([
    test_df_tabm.drop(columns='id').add_prefix('tabm_'),
    test_df_lgbm.drop(columns='id').add_prefix('lgbm_'),
    test_df_xgb.drop(columns='id').add_prefix('xgb_'),
    test_df_mlp.drop(columns='id').add_prefix('mlp_'),
    test_df_tabm_res.drop(columns='id').add_prefix('tabm_res_'),
    # test_df_xgb2.drop(columns=['id', 'Unnamed: 0']).add_prefix('xgb2_'),
    test_df_cat2.drop(columns=['id', 'Unnamed: 0']).add_prefix('cat2_'),
    test_df_lgb2.drop(columns='id').add_prefix('lgb2_'),
    test_df_residuals['WeightedEnsemble_L5_res'],
], axis=1)

# oofs_df = pd.concat([oofs_df_baseline, oofs_df_residuals], axis=1)
# test_df = pd.concat([test_df_baseline, test_df_residuals], axis=1)

train = pd.read_csv('/kaggle/input/playground-series-s5e10/train.csv')
y = train['accident_risk']

In [5]:
TARGET = 'accident_risk'
FEATURES = [col for col in oofs_df.columns if col!='accident_risk']

In [6]:
oofs_df = pd.concat([oofs_df, y], axis=1)

In [7]:
# train = train.fillna(0)
# test = test.fillna(0)

In [8]:
import shutil, os
old = '/kaggle/working/AutogluonModels'
if os.path.exists(old):
    shutil.rmtree(old)

In [9]:
import autogluon.core.utils.utils as core_utils
from sklearn.model_selection import RepeatedKFold, RepeatedStratifiedKFold, LeaveOneGroupOut

_ORIG_CVSPLITTER_INIT = core_utils.CVSplitter.__init__

def _cvsplitter_init_with_42(self, splitter_cls=None, n_splits=5, n_repeats=1,
                             random_state=None, stratify=False, bin=False,
                             n_bins=None, groups=None):
    # force our seed, ignore the 0 that the trainer passes
    _ORIG_CVSPLITTER_INIT(
        self,
        splitter_cls=splitter_cls,
        n_splits=n_splits,
        n_repeats=n_repeats,
        random_state=42,        # <-- your seed
        stratify=stratify,
        bin=bin,
        n_bins=n_bins,
        groups=groups,
    )

core_utils.CVSplitter.__init__ = _cvsplitter_init_with_42

In [10]:
from autogluon.tabular import TabularPredictor
import warnings
warnings.filterwarnings('ignore')

# PEAK_XGB = {
#     'n_estimators': 100_000, 'learning_rate': 0.01, 'max_depth': 6,
#     'subsample': 0.9, 'colsample_bytree': 0.6, 'reg_alpha': 0.0, 'reg_lambda': 0.0,
#     'tree_method': 'gpu_hist', 'device': 'cuda', 'n_jobs': -1, 'verbosity': 0
# }

# PEAK_LGB = {
#     'n_estimators': 100_000, 'learning_rate': 0.01, 'num_leaves': 64,
#     'subsample': 0.9, 'colsample_bytree': 0.6, 'reg_alpha': 0.0, 'reg_lambda': 0.0,
#     'device': 'gpu', 'n_jobs': -1, 'verbosity': -1
# }

# PEAK_CAT = {
#     'iterations': 100_000, 'learning_rate': 0.01, 'depth': 6,
#     'l2_leaf_reg': 0.0, 'subsample': 0.9, 'task_type': 'GPU', 'verbose': False
# }

# ----------  AutoGluon search space  ----------
predictor = TabularPredictor(
    label=TARGET,
    eval_metric='rmse',
    problem_type='regression',
    path='AutogluonModels/full_hpo'
).fit(
    train_data=oofs_df,
    time_limit=39600,  # 2 hours for extensive HPO
    presets='best_quality',
    num_bag_folds=5,
    num_stack_levels=3,
    num_bag_sets=3,
    auto_stack=True,
    raise_on_no_models_fitted=False,
    # REMOVED: hyperparameter_tune=True,  # Not needed - just use hyperparameter_tune_kwargs
    # hyperparameter_tune_kwargs={
    #     'scheduler': 'local',
    #     'searcher': 'bayesopt',
    #     'num_trials': 50,
    # },
    # hyperparameters={
    #     # XGBoost with search space
    #     'XGB': {
    #         'n_estimators': 10000,
    #         'learning_rate': [0.005, 0.01, 0.02, 0.05, 0.1],
    #         'max_depth': [4, 5, 6, 7, 8, 9],
    #         'subsample': [0.7, 0.8, 0.9, 1.0],
    #         'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    #         'reg_alpha': [0, 0.1, 0.5, 1.0],
    #         'reg_lambda': [0, 0.1, 0.5, 1.0, 5.0],
    #         'min_child_weight': [1, 3, 5, 7],
    #         'tree_method': 'gpu_hist',
    #         'device': 'cuda',
    #     },
        
    #     # LightGBM with search space
    #     'GBM': {
    #         'n_estimators': 10000,
    #         'learning_rate': [0.005, 0.01, 0.02, 0.05, 0.1],
    #         'num_leaves': [31, 63, 127, 255],
    #         'max_depth': [6, 8, 10, 12, -1],
    #         'subsample': [0.7, 0.8, 0.9, 1.0],
    #         'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    #         'reg_alpha': [0, 0.1, 0.5, 1.0],
    #         'reg_lambda': [0, 0.1, 0.5, 1.0, 5.0],
    #         'min_child_samples': [5, 10, 20, 30],
    #         'device': 'gpu',
    #     },
        
    #     # CatBoost with search space
    #     'CAT': {
    #         'iterations': 10000,
    #         'learning_rate': [0.005, 0.01, 0.02, 0.05, 0.1],
    #         'depth': [4, 5, 6, 7, 8, 9],
    #         'l2_leaf_reg': [1, 3, 5, 7, 9],
    #         'random_strength': [0.1, 0.5, 1.0, 2.0],
    #         'bagging_temperature': [0, 0.5, 1.0],
    #         'subsample': [0.7, 0.8, 0.9, 1.0],
    #         'task_type': 'GPU',
    #     },
        
    #     # Neural networks with tuning
    #     'NN_TORCH': {
    #         'num_layers': [2, 3, 4],
    #         'hidden_size': [128, 256, 512],
    #         'dropout_prob': [0.0, 0.1, 0.2, 0.3],
    #         'learning_rate': [1e-4, 1e-3, 1e-2],
    #         'num_epochs': [50, 100, 150],
    #         'activation': ['relu', 'elu', 'tanh', 'leaky_relu'],
    #         'use_batchnorm': [True, False],
    #     },
        
        # # Random Forest with tuning
        # 'RF': {
        #     'n_estimators': [100, 200, 300, 500],
        #     'max_depth': [10, 15, 20, None],
        #     'max_features': [0.5, 0.7, 0.9, 'auto', 'sqrt'],
        #     'min_samples_split': [2, 5, 10],
        #     'min_samples_leaf': [1, 2, 4],
        #     'bootstrap': [True, False],
        # },
        
        # # Extra Trees with tuning
        # 'XT': {
        #     'n_estimators': [100, 200, 300, 500],
        #     'max_depth': [10, 15, 20, None],
        #     'max_features': [0.5, 0.7, 0.9, 'auto', 'sqrt'],
        #     'min_samples_split': [2, 5, 10],
        #     'min_samples_leaf': [1, 2, 4],
        #     'bootstrap': [True, False],
        # },
        
        # # KNN with tuning
        # 'KNN': {
        #     'n_neighbors': [3, 5, 7, 10, 15, 20, 30, 50],
        #     'weights': ['uniform', 'distance'],
        #     'metric': ['euclidean', 'minkowski', 'manhattan'],
        # },
        
        # # Linear models with tuning
        # 'LR': {
        #     'fit_intercept': [True, False],
        #     'normalize': [True, False],
        #     'alpha': [0.0001, 0.001, 0.01, 0.1, 1.0, 10.0],
        # }
    # },
    verbosity=2,
    num_gpus=1
)

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.4.0
Python Version:     3.11.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Sun Nov 10 10:07:59 UTC 2024
CPU Count:          4
Memory Avail:       29.72 GB / 31.35 GB (94.8%)
Disk Space Avail:   19.50 GB / 19.52 GB (99.9%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=3, num_bag_folds=5, num_bag_sets=3
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked overfitting and enable or disable stacking as a consequence.
	This is used to identify the optimal `num_stack_levels` value. Copies of AutoGluon will be fit on subsets of the da

[1000]	valid_set's rmse: 0.05559
[2000]	valid_set's rmse: 0.0555809
[3000]	valid_set's rmse: 0.0555799
[1000]	valid_set's rmse: 0.0561631
[2000]	valid_set's rmse: 0.056149
[3000]	valid_set's rmse: 0.056143
[4000]	valid_set's rmse: 0.0561422
[5000]	valid_set's rmse: 0.056142
[1000]	valid_set's rmse: 0.0562718
[2000]	valid_set's rmse: 0.0562581
[3000]	valid_set's rmse: 0.0562518
[4000]	valid_set's rmse: 0.0562502
[5000]	valid_set's rmse: 0.056248
[6000]	valid_set's rmse: 0.0562485
[7000]	valid_set's rmse: 0.0562479
[8000]	valid_set's rmse: 0.0562474
[1000]	valid_set's rmse: 0.0558893
[2000]	valid_set's rmse: 0.0558771
[3000]	valid_set's rmse: 0.0558742
[4000]	valid_set's rmse: 0.0558743
[5000]	valid_set's rmse: 0.0558728
[6000]	valid_set's rmse: 0.0558732
[1000]	valid_set's rmse: 0.05598
[2000]	valid_set's rmse: 0.0559692
[3000]	valid_set's rmse: 0.0559644
[4000]	valid_set's rmse: 0.0559645
[1000]	valid_set's rmse: 0.0561553
[2000]	valid_set's rmse: 0.0561427
[3000]	valid_set's rmse: 0.0

	-0.0559	 = Validation score   (-root_mean_squared_error)
	816.85s	 = Training   runtime
	272.07s	 = Validation runtime
Fitting model: LightGBM_BAG_L1 ... Training model for up to 2200.76s of the 8797.28s of remaining time.
Specified total num_gpus: 1, but only 0 are available. Will use 0 instead
	Fitting 15 child models (S1F1 - S3F5) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=2, gpus=0)
	-0.0559	 = Validation score   (-root_mean_squared_error)
	48.28s	 = Training   runtime
	6.21s	 = Validation runtime
Fitting model: RandomForestMSE_BAG_L1 ... Training model for up to 2145.57s of the 8742.08s of remaining time.
Specified total num_gpus: 1, but only 0 are available. Will use 0 instead
	-0.0563	 = Validation score   (-root_mean_squared_error)
	1038.92s	 = Training   runtime
	21.41s	 = Validation runtime
Fitting model: CatBoost_BAG_L1 ... Training model for up to 1083.11s of the 7679.62s of remaining time.
Specified total num_gpus: 1, but only 0 are available. Wil

[1000]	valid_set's rmse: 0.0555884
[1000]	valid_set's rmse: 0.0561385
[2000]	valid_set's rmse: 0.0561369
[1000]	valid_set's rmse: 0.0562604
[2000]	valid_set's rmse: 0.0562514
[3000]	valid_set's rmse: 0.0562465
[4000]	valid_set's rmse: 0.0562411
[5000]	valid_set's rmse: 0.056238
[6000]	valid_set's rmse: 0.0562362
[7000]	valid_set's rmse: 0.0562309
[8000]	valid_set's rmse: 0.0562285
[9000]	valid_set's rmse: 0.0562247
[10000]	valid_set's rmse: 0.0562205
[1000]	valid_set's rmse: 0.0558644
[2000]	valid_set's rmse: 0.0558574
[3000]	valid_set's rmse: 0.0558551
[4000]	valid_set's rmse: 0.0558544
[5000]	valid_set's rmse: 0.0558518
[6000]	valid_set's rmse: 0.0558501
[7000]	valid_set's rmse: 0.0558488
[8000]	valid_set's rmse: 0.0558476
[9000]	valid_set's rmse: 0.0558484
[10000]	valid_set's rmse: 0.055848
[1000]	valid_set's rmse: 0.0559538
[1000]	valid_set's rmse: 0.0561321
[2000]	valid_set's rmse: 0.0561308
[3000]	valid_set's rmse: 0.0561287
[4000]	valid_set's rmse: 0.0561295
[1000]	valid_set's r

	-0.0559	 = Validation score   (-root_mean_squared_error)
	883.82s	 = Training   runtime
	289.04s	 = Validation runtime
Fitting model: LightGBM_BAG_L2 ... Training model for up to 1754.94s of the 5419.88s of remaining time.
Specified total num_gpus: 1, but only 0 are available. Will use 0 instead
	Fitting 15 child models (S1F1 - S3F5) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=2, gpus=0)
	-0.0559	 = Validation score   (-root_mean_squared_error)
	82.94s	 = Training   runtime
	9.27s	 = Validation runtime
Fitting model: RandomForestMSE_BAG_L2 ... Training model for up to 1661.70s of the 5326.65s of remaining time.
Specified total num_gpus: 1, but only 0 are available. Will use 0 instead
	-0.0559	 = Validation score   (-root_mean_squared_error)
	1568.73s	 = Training   runtime
	18.13s	 = Validation runtime
Fitting model: CatBoost_BAG_L2 ... Training model for up to 72.84s of the 3737.79s of remaining time.
Specified total num_gpus: 1, but only 0 are available. Will 

...

	-0.0559	 = Validation score   (-root_mean_squared_error)
	489.63s	 = Training   runtime
	151.03s	 = Validation runtime
Fitting model: LightGBM_BAG_L4 ... Training model for up to 576.81s of the 576.77s of remaining time.
Specified total num_gpus: 1, but only 0 are available. Will use 0 instead
	Fitting 15 child models (S1F1 - S3F5) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=2, gpus=0)
	-0.0558	 = Validation score   (-root_mean_squared_error)
	56.87s	 = Training   runtime
	5.67s	 = Validation runtime
Fitting model: RandomForestMSE_BAG_L4 ... Training model for up to 513.48s of the 513.44s of remaining time.
Specified total num_gpus: 1, but only 0 are available. Will use 0 instead
	-0.0564	 = Validation score   (-root_mean_squared_error)
	507.76s	 = Training   runtime
	7.37s	 = Validation runtime
Fitting model: WeightedEnsemble_L5 ... Training model for up to 360.00s of the -2.84s of remaining time.
Specified total num_gpus: 1, but only 0 are available. Will use

[1000]	valid_set's rmse: 0.0561098
[2000]	valid_set's rmse: 0.0560998
[3000]	valid_set's rmse: 0.0560971
[4000]	valid_set's rmse: 0.0560983
[1000]	valid_set's rmse: 0.0559762
[2000]	valid_set's rmse: 0.0559548
[3000]	valid_set's rmse: 0.0559496
[4000]	valid_set's rmse: 0.0559486
[1000]	valid_set's rmse: 0.056058
[2000]	valid_set's rmse: 0.0560506
[3000]	valid_set's rmse: 0.0560509
[1000]	valid_set's rmse: 0.0558774
[2000]	valid_set's rmse: 0.0558665
[3000]	valid_set's rmse: 0.0558653
[1000]	valid_set's rmse: 0.0558152
[2000]	valid_set's rmse: 0.0558001
[3000]	valid_set's rmse: 0.0557958
[4000]	valid_set's rmse: 0.0557939
[5000]	valid_set's rmse: 0.0557944
[1000]	valid_set's rmse: 0.05628
[2000]	valid_set's rmse: 0.0562686
[3000]	valid_set's rmse: 0.0562688
[1000]	valid_set's rmse: 0.0559423
[2000]	valid_set's rmse: 0.055932
[3000]	valid_set's rmse: 0.0559263
[4000]	valid_set's rmse: 0.0559244
[5000]	valid_set's rmse: 0.0559247
[6000]	valid_set's rmse: 0.0559242
[7000]	valid_set's rmse:

	-0.0559	 = Validation score   (-root_mean_squared_error)
	920.29s	 = Training   runtime
	322.39s	 = Validation runtime
Fitting model: LightGBM_BAG_L1 ... Training model for up to 27694.23s of the 27694.22s of remaining time.
Specified total num_gpus: 1, but only 0 are available. Will use 0 instead
	Fitting 15 child models (S1F1 - S3F5) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=2, gpus=0)
	-0.0559	 = Validation score   (-root_mean_squared_error)
	56.91s	 = Training   runtime
	7.7s	 = Validation runtime
Fitting model: RandomForestMSE_BAG_L1 ... Training model for up to 27628.81s of the 27628.79s of remaining time.
Specified total num_gpus: 1, but only 0 are available. Will use 0 instead
	-0.0562	 = Validation score   (-root_mean_squared_error)
	1224.15s	 = Training   runtime
	23.24s	 = Validation runtime
Fitting model: CatBoost_BAG_L1 ... Training model for up to 26378.93s of the 26378.91s of remaining time.
Specified total num_gpus: 1, but only 0 are available

[1000]	valid_set's rmse: 0.0559135
[1000]	valid_set's rmse: 0.0554089
[1000]	valid_set's rmse: 0.0561888


	-0.0559	 = Validation score   (-root_mean_squared_error)
	195.46s	 = Training   runtime
	49.49s	 = Validation runtime
Fitting model: NeuralNetFastAI_r191_BAG_L1 ... Training model for up to 8834.25s of the 8834.23s of remaining time.
Specified total num_gpus: 1, but only 0 are available. Will use 0 instead
	Fitting 15 child models (S1F1 - S3F5) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=2, gpus=0)
	Ran out of time, stopping training early. (Stopping on epoch 15)
	Ran out of time, stopping training early. (Stopping on epoch 15)
	Ran out of time, stopping training early. (Stopping on epoch 16)
	Ran out of time, stopping training early. (Stopping on epoch 16)
	Ran out of time, stopping training early. (Stopping on epoch 17)
	Ran out of time, stopping training early. (Stopping on epoch 17)
	Ran out of time, stopping training early. (Stopping on epoch 17)
	Ran out of time, stopping training early. (Stopping on epoch 17)
	Ran out of time, stopping training early. (S

In [11]:
leaderboard = predictor.leaderboard()
leaderboard

,model,score_val,eval_metric,pred_time_val,fit_time,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,WeightedEnsemble_L2,-0.055851,root_mean_squared_error,40.656674,12310.847346,0.009683,0.867202,2,True,15
1,NeuralNetFastAI_BAG_L1,-0.055862,root_mean_squared_error,10.182856,3705.638846,10.182856,3705.638846,1,True,6
2,LightGBM_BAG_L1,-0.055887,root_mean_squared_error,7.699641,56.909141,7.699641,56.909141,1,True,2
3,CatBoost_r9_BAG_L1,-0.055891,root_mean_squared_error,6.733197,466.128770,6.733197,466.128770,1,True,14
4,LightGBM_r131_BAG_L1,-0.055893,root_mean_squared_error,49.489526,195.462010,49.489526,195.462010,1,True,12
5,CatBoost_BAG_L1,-0.055895,root_mean_squared_error,0.653742,390.696394,0.653742,390.696394,1,True,4
6,NeuralNetTorch_r79_BAG_L1,-0.055899,root_mean_squared_error,5.531639,8454.940795,5.531639,8454.940795,1,True,11
7,CatBoost_r177_BAG_L1,-0.055900,root_mean_squared_error,0.509261,281.467655,0.509261,281.467655,1,True,10
8,XGBoost_BAG_L1,-0.055907,root_mean_squared_error,2.829684,43.571361,2.829684,43.571361,1,True,7
9,LightGBMLarge_BAG_L1,-0.055910,root_mean_squared_error,13.771741,96.341167,13.771741,96.341167,1,True,9


In [12]:
models = leaderboard['model'].to_list()
oofs_dict = {}
for m in models:
    oofs_dict[m] = predictor.predict_oof(model=m)

oofs_df = pd.DataFrame(oofs_dict)

In [13]:
predictor2 = TabularPredictor.load('/kaggle/working/AutogluonModels/full_hpo')
test_preds = {}
for m in models:
    test_preds[m] = predictor2.predict(test_df, model=m)

test_preds_df = pd.DataFrame(test_preds)

In [14]:
# for col in oofs_df.columns:
#     oofs_df[col] = oofs_df[col] + train['y']
#     test_preds_df[col] = test_preds_df[col] + test['y']

In [15]:
samp = pd.read_csv('/kaggle/input/playground-series-s5e10/sample_submission.csv')
samp['accident_risk'] = test_preds_df.iloc[:,0]
samp.to_csv('autogluon_meta_8_models.csv', index=False)

In [16]:
leaderboard.to_csv('leaderboard_autogluon_residuals.csv', index=False)
oofs_df.to_csv('oofs_autogluon_residuals.csv', index=False)
test_preds_df.to_csv('test_preds_autogluon_residuals.csv', index=False)